# LIMUC ResNet-50 (frozen) + Logistic Regression
Extract frozen ResNet-50 features and train a linear classifier.


In [1]:
import os
import json
import random
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


In [2]:
# Paths & config
def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
LABEL_MAP_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "label_map.csv"
OUT_DIR = DATA_ROOT / "1_frozen_encoders" / "results" / "resnet50_frozen_logreg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.getenv("SEED", "42"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "16"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "0"))
MAX_SAMPLES = int(os.getenv("MAX_SAMPLES", "0")) or None
CACHE_FEATURES = os.getenv("CACHE_FEATURES", "1") == "1"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Data root:", DATA_ROOT)
print("Output dir:", OUT_DIR)



Device: cuda
Data root: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/LIMUC
Output dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/LIMUC/1_frozen_encoders/out/resnet50_frozen_logreg


In [3]:
# Seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Load metadata
meta = pd.read_csv(META_CSV)

# Normalize image paths to absolute (base is 0_dataset_prep)
images_base = DATA_ROOT / "0_dataset_prep"

def to_abs(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return (images_base / p).resolve()

meta["image_path"] = meta["image_path"].apply(lambda p: str(to_abs(p)))

# Label map
if LABEL_MAP_CSV.exists():
    label_map = pd.read_csv(LABEL_MAP_CSV)
    id_to_name = dict(zip(label_map.label_id, label_map.label_name))
else:
    id_to_name = {i: name for i, name in enumerate(sorted(meta.label_name.unique()))}

name_to_id = {v: k for k, v in id_to_name.items()}
meta["label_id"] = meta["label_name"].map(name_to_id)

# Keep only rows with existing images
meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

# Optionally subsample
if MAX_SAMPLES:
    meta = meta.sample(n=min(MAX_SAMPLES, len(meta)), random_state=SEED).reset_index(drop=True)

print("Rows:", len(meta))
print("Splits:\n", meta["split"].value_counts())

Rows: 11276
Splits:
 split
train    8669
test     1686
val       921
Name: count, dtype: int64


In [4]:
class ImageDS(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["image_path"].tolist()
        self.labels = df["label_id"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        label = int(self.labels[idx])
        return img, label


# ResNet-50 ImageNet transforms
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [5]:
# Build feature extractor
base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
backbone = nn.Sequential(*list(base.children())[:-1]).to(DEVICE)
backbone.eval()

def extract_features(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    ds = ImageDS(df, train_tf)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    feats, labels = [], []
    with torch.no_grad():
        for x, y in dl:
            x = x.to(DEVICE)
            out = backbone(x).squeeze(-1).squeeze(-1)
            feats.append(out.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats, axis=0), np.concatenate(labels, axis=0)


In [6]:
# Extract / cache features
def _load_or_extract(split_name: str, df: pd.DataFrame):
    feat_path = OUT_DIR / f"features_{split_name}.npy"
    label_path = OUT_DIR / f"labels_{split_name}.npy"
    if CACHE_FEATURES and feat_path.exists() and label_path.exists():
        X = np.load(feat_path)
        y = np.load(label_path)
        return X, y
    X, y = extract_features(df)
    if CACHE_FEATURES:
        np.save(feat_path, X)
        np.save(label_path, y)
    return X, y

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"].isin(["val", "validation"])].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

X_train, y_train = _load_or_extract("train", train_df)
X_val, y_val = _load_or_extract("val", val_df)
X_test, y_test = _load_or_extract("test", test_df)

print("Train feats:", X_train.shape)



Train feats: (8669, 2048)


In [7]:
# Train Logistic Regression
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
        multi_class="multinomial",
    )),
])

clf.fit(X_train, y_train)

# Predictions
train_pred = clf.predict(X_train)
val_pred = clf.predict(X_val)
test_pred = clf.predict(X_test)

train_prob = clf.predict_proba(X_train)
val_prob = clf.predict_proba(X_val)
test_prob = clf.predict_proba(X_test)


/home/arcturus/miniforge3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [8]:
# =====================
# Metrics helpers
# =====================
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)

try:
    from scipy.stats import spearmanr
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False


def expected_calibration_error(y_true, y_prob, n_bins=10):
    if y_prob is None:
        return None
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        if mask.any():
            ece += abs(accuracies[mask].mean() - confidences[mask].mean()) * mask.mean()
    return float(ece)


def compute_metrics(y_true, y_pred, labels, label_names, y_prob=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    summary = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "qwk": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

    if _HAS_SCIPY:
        summary["spearman"] = float(spearmanr(y_true, y_pred).correlation)

    if y_prob is not None:
        try:
            summary["auroc_ovr"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr"))
        except Exception:
            summary["auroc_ovr"] = None
        summary["ece"] = expected_calibration_error(y_true, y_prob, n_bins=10)

    return summary, report


def save_split_outputs(
    split_name,
    y_true,
    y_pred,
    labels,
    label_names,
    out_dir,
    y_prob=None,
    df_meta=None,
):
    summary, report = compute_metrics(y_true, y_pred, labels, label_names, y_prob)

    # Save metrics
    metrics = {
        "split": split_name,
        "summary": summary,
        "report": report,
    }
    with open(out_dir / f"metrics_{split_name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # Save per-class report
    per_class = {k: v for k, v in report.items() if k in label_names}
    pd.DataFrame(per_class).T.to_csv(out_dir / f"per_class_{split_name}.csv")

    # Save predictions
    pred_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
    })
    if df_meta is not None:
        pred_df["img_id"] = df_meta["img_id"].values
        pred_df["image_path"] = df_meta["image_path"].values
    if y_prob is not None:
        for i, name in enumerate(label_names):
            pred_df[f"prob_{name}"] = y_prob[:, i]
    pred_df.to_csv(out_dir / f"pred_{split_name}.csv", index=False)

    return summary, report


In [9]:
labels = sorted(id_to_name.keys())
label_names = [id_to_name[i] for i in labels]

# Save metrics and predictions
train_summary, _ = save_split_outputs("train", y_train, train_pred, labels, label_names, OUT_DIR, train_prob, train_df)
val_summary, _ = save_split_outputs("val", y_val, val_pred, labels, label_names, OUT_DIR, val_prob, val_df)
test_summary, _ = save_split_outputs("test", y_test, test_pred, labels, label_names, OUT_DIR, test_prob, test_df)

print("Train summary:")
print(json.dumps(train_summary, indent=2))
print("Val summary:")
print(json.dumps(val_summary, indent=2))
print("Test summary:")
print(json.dumps(test_summary, indent=2))

# Confusion matrices
val_cm = confusion_matrix(y_val, val_pred, labels=labels)
test_cm = confusion_matrix(y_test, test_pred, labels=labels)
np.save(OUT_DIR / "confusion_val.npy", val_cm)
np.save(OUT_DIR / "confusion_test.npy", test_cm)

# Plot
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

for split_name, cm in [("val", val_cm), ("test", test_cm)]:
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(include_values=False, cmap="Blues", ax=ax, xticks_rotation=90)
    plt.title(f"{split_name.upper()} Confusion Matrix (ResNet50 frozen + LR)")
    plt.tight_layout()
    fig_path = OUT_DIR / f"confusion_{split_name}.png"
    plt.savefig(fig_path, dpi=200)
    plt.close(fig)

# Run meta
from datetime import datetime, timezone
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = os.getenv("RUN_ID") or f"{OUT_DIR.name}_{RUN_TIMESTAMP_UTC.replace(':', '').replace('-', '')}"

run_meta = {
    "model": "resnet50_frozen_logreg",
    "seed": SEED,
    "split_hash": (DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "split_hash.txt").read_text().strip()
        if (DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "split_hash.txt").exists() else None,
    "run_id": RUN_ID,
    "timestamp_utc": RUN_TIMESTAMP_UTC,
    "out_dir": str(OUT_DIR),
    "notebook_path": str(Path.cwd()),

}
with open(OUT_DIR / "run_meta.json", "w") as f:
    json.dump(run_meta, f, indent=2)

print("Saved outputs to", OUT_DIR)




Train summary:
{
  "accuracy": 0.9518975660399124,
  "balanced_accuracy": 0.9731332432211026,
  "macro_f1": 0.9671209656578799,
  "weighted_f1": 0.9524426112588847,
  "qwk": 0.9725222168657247,
  "mae": 0.04810243396008767,
  "rmse": 0.21932267087578444,
  "spearman": 0.9441992000917259,
  "auroc_ovr": 0.9945773435412634,
  "ece": 0.023128653787304117
}
Val summary:
{
  "accuracy": 0.6623235613463626,
  "balanced_accuracy": 0.5823288902880663,
  "macro_f1": 0.5817344461585764,
  "weighted_f1": 0.6667006423248452,
  "qwk": 0.7382795508620442,
  "mae": 0.38436482084690554,
  "rmse": 0.7036433802109261,
  "spearman": 0.6938119342586714,
  "auroc_ovr": 0.8454708453797303,
  "ece": 0.27775887573290126
}
Test summary:
{
  "accuracy": 0.6198102016607354,
  "balanced_accuracy": 0.5419914497343633,
  "macro_f1": 0.5346085102016398,
  "weighted_f1": 0.6277139024236273,
  "qwk": 0.6833679994975561,
  "mae": 0.43238434163701067,
  "rmse": 0.7366848894435934,
  "spearman": 0.6160496693686127,
  "au